# Strategic Plan -> Top-3 Funding Calls (Phase 1)

This notebook implements the Phase 1 baseline: for each strategic plan, retrieve candidate chunks from Chroma, aggregate at funding-call level, and return a top-3 ranking.


## Setup

Install required packages in the active environment.


In [28]:
%pip install -q chromadb pypdf sentence-transformers pandas tqdm


Note: you may need to restart the kernel to use updated packages.


## Imports

Import libraries for PDF loading, Chroma retrieval, and ranking output.


In [29]:
from pathlib import Path
import re
from typing import Dict, List

import chromadb
import pandas as pd
from chromadb.utils import embedding_functions
from pypdf import PdfReader
from tqdm.auto import tqdm

from pipeline_core_methods import (
    build_call_xai as core_build_call_xai,
    chunk_text as core_chunk_text,
    clean_text as core_clean_text,
    rank_calls_for_sp_query as core_rank_calls_for_sp_query,
    similarity_from_distance as core_similarity_from_distance,
)


## Configuration

Centralize paths, collection/model settings, and ranking constants for reproducible runs.


In [ ]:
PROJECT_ROOT = Path.cwd()
CHROMA_DIR = PROJECT_ROOT / "data" / "chroma"
COLLECTION_NAME = "funding_calls"
EMBED_MODEL = "Qwen/Qwen3-Embedding-0.6B"

STRATEGIC_PLAN_DIR = PROJECT_ROOT / "docs" / "strategicplans"
EXPECTED_SP_FILES = {
    "Naples_Federico_Piano_strategico_2021_2026.pdf",
    "Politechnico_di_Milano_2023-2025.pdf",
    "Piano Strategico Luiss 2021-2024 (per sito).pdf",
    "Bocconi_Piano_Strategico2021-2025&Vision2030.pdf",
    "LUM_PIANO-STRATEGICO-DATENEO-2021-2025.pdf",
    "Sapienza_2021_2027.pdf",
}

N_RESULTS = 30
TOP_K_CALLS = 3
EVIDENCE_PER_CALL = 3
STRONG_HIT_THRESHOLD = 0.55

SP_CHUNK_SIZE = 1200
SP_CHUNK_OVERLAP = 200
SP_MIN_CHUNK_CHARS = 160

print(f"Project root: {PROJECT_ROOT}")
print(f"Chroma dir: {CHROMA_DIR}")
print(f"Collection: {COLLECTION_NAME}")
print(f"SP directory: {STRATEGIC_PLAN_DIR}")
print(f"SP chunking: size={SP_CHUNK_SIZE}, overlap={SP_CHUNK_OVERLAP}, min_chars={SP_MIN_CHUNK_CHARS}")


Project root: /home/lburmazevic/Projects/EY-AI-Techniques-Solution
Chroma dir: /home/lburmazevic/Projects/EY-AI-Techniques-Solution/data/chroma
Collection: funding_calls
SP directory: /home/lburmazevic/Projects/EY-AI-Techniques-Solution/docs/strategicplans


## Strategic Plans

Load the six strategic plan PDFs, clean text, and split each SP into overlapping chunks for retrieval queries.


In [31]:
def clean_text(text: str) -> str:
    return core_clean_text(text)


def chunk_text(text: str, size: int = SP_CHUNK_SIZE, overlap: int = SP_CHUNK_OVERLAP) -> List[str]:
    return core_chunk_text(text, size=size, overlap=overlap, min_chars=SP_MIN_CHUNK_CHARS)


def extract_sp_text(pdf_path: Path) -> Dict:
    reader = PdfReader(str(pdf_path))
    page_texts: List[str] = []

    for page in reader.pages:
        text = clean_text(page.extract_text() or "")
        if len(text) >= 40:
            page_texts.append(text)

    sp_text = "\n".join(page_texts)
    sp_chunks = chunk_text(sp_text)

    return {
        "sp_id": pdf_path.stem,
        "source_file": pdf_path.name,
        "text": sp_text,
        "chunks": sp_chunks,
        "non_empty_pages": len(page_texts),
    }


sp_paths = sorted(STRATEGIC_PLAN_DIR.glob("*.pdf"))
found = {p.name for p in sp_paths}
missing = EXPECTED_SP_FILES - found
if missing:
    raise FileNotFoundError(f"Missing strategic plan files: {sorted(missing)}")

strategic_plans = []
for path in tqdm(sp_paths, desc="Loading strategic plans"):
    if path.name in EXPECTED_SP_FILES:
        strategic_plans.append(extract_sp_text(path))

print(f"Loaded {len(strategic_plans)} strategic plans")
for sp in strategic_plans:
    print(f" - {sp['source_file']}: non_empty_pages={sp['non_empty_pages']}, chars={len(sp['text'])}, chunks={len(sp['chunks'])}")



Loading strategic plans: 100%|██████████| 6/6 [00:13<00:00,  2.27s/it]

Loaded 6 strategic plans
 - Bocconi_Piano_Strategico2021-2025&Vision2030.pdf: non_empty_pages=38, chars=85284
 - LUM_PIANO-STRATEGICO-DATENEO-2021-2025.pdf: non_empty_pages=74, chars=195991
 - Naples_Federico_Piano_strategico_2021_2026.pdf: non_empty_pages=68, chars=68387
 - Piano Strategico Luiss 2021-2024 (per sito).pdf: non_empty_pages=62, chars=55615
 - Politechnico_di_Milano_2023-2025.pdf: non_empty_pages=29, chars=29591
 - Sapienza_2021_2027.pdf: non_empty_pages=31, chars=58569


## Connect to Chroma

Attach to the persistent ChromaDB collection generated during ingestion.


In [32]:
client = chromadb.PersistentClient(path=str(CHROMA_DIR))
embed_fn = embedding_functions.SentenceTransformerEmbeddingFunction(model_name=EMBED_MODEL)
collection = client.get_collection(name=COLLECTION_NAME, embedding_function=embed_fn)
print("Chroma collection is ready")


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1203.78it/s]


Chroma collection is ready


## Phase 1 Ranking Logic

For each SP chunk, retrieve candidate chunks from Chroma, aggregate by funding call across all SP chunks, compute weighted scores, and keep top-3 calls.


In [33]:
def similarity_from_distance(distance: float) -> float:
    return core_similarity_from_distance(distance)


def rank_calls_for_sp(sp: Dict, n_results: int = N_RESULTS) -> Dict:
    return core_rank_calls_for_sp_query(
        collection=collection,
        sp=sp,
        n_results=n_results,
        strong_hit_threshold=STRONG_HIT_THRESHOLD,
        evidence_per_call=EVIDENCE_PER_CALL,
        top_k_calls=TOP_K_CALLS,
    )



## Batch Run

Run the ranking pipeline for all SPs and return a compact table of top recommendations.


In [34]:
phase1_results = []
for sp in tqdm(strategic_plans, desc='Ranking funding calls'):
    phase1_results.append(rank_calls_for_sp(sp))

rows = []
for result in phase1_results:
    for idx, call in enumerate(result['top_calls'], start=1):
        rows.append({
            'sp_file': result['sp_file'],
            'rank': idx,
            'call_name': call['call_name'],
            'final_score': call['final_score'],
            'semantic_score': call['semantic_score'],
            'coverage_score': call['coverage_score'],
            'consistency_score': call['consistency_score'],
        })

ranking_df = pd.DataFrame(rows).sort_values(['sp_file', 'rank']).reset_index(drop=True)
ranking_df


Ranking funding calls: 100%|██████████| 6/6 [00:01<00:00,  3.61it/s]


,sp_file,rank,call_name,final_score,semantic_score,coverage_score,consistency_score
0,Bocconi_Piano_Strategico2021-2025&Vision2030.pdf,1,HORIZON-WIDERA-2023-ERA-01-06.pdf,0.6571,0.7101,0.5000,0.6
1,Bocconi_Piano_Strategico2021-2025&Vision2030.pdf,2,HORIZON-WIDERA-2022-ERA-01-10.pdf,0.6067,0.7144,0.3333,0.4
2,Bocconi_Piano_Strategico2021-2025&Vision2030.pdf,3,HORIZON-WIDERA-2021-ERA-01-20.pdf,0.6067,0.7143,0.3333,0.4
3,LUM_PIANO-STRATEGICO-DATENEO-2021-2025.pdf,1,Partenariati Estesi PNRR.pdf,0.7750,0.6786,1.0000,1.0
4,LUM_PIANO-STRATEGICO-DATENEO-2021-2025.pdf,2,PRIN 2022 PNRR.pdf,0.7747,0.6781,1.0000,1.0
5,LUM_PIANO-STRATEGICO-DATENEO-2021-2025.pdf,3,Ecosistemi dell'Innovazione PNRR.pdf,0.7487,0.6886,0.8333,1.0
6,Naples_Federico_Piano_strategico_2021_2026.pdf,1,PRIN 2022 PNRR.pdf,0.7536,0.6480,1.0000,1.0
7,Naples_Federico_Piano_strategico_2021_2026.pdf,2,Ecosistemi dell'Innovazione PNRR.pdf,0.7241,0.6534,0.8333,1.0
8,Naples_Federico_Piano_strategico_2021_2026.pdf,3,PRIN PNRR 2022 NextGen.pdf,0.7236,0.6527,0.8333,1.0
9,Piano Strategico Luiss 2021-2024 (per sito).pdf,1,Ecosistemi dell'Innovazione PNRR.pdf,0.8192,0.7418,1.0000,1.0


## Evidence Preview

Inspect the top supporting chunks behind each recommended call for quick explainability checks.


In [35]:
def print_evidence_for_sp(sp_file: str):
    matches = [x for x in phase1_results if x['sp_file'] == sp_file]
    if not matches:
        print(f'No result for {sp_file}')
        return

    result = matches[0]
    print(f"Strategic plan: {result['sp_file']}")
    for i, call in enumerate(result['top_calls'], start=1):
        print(f"\nRank {i}: {call['call_name']} | final_score={call['final_score']}")
        for ev in call['evidence']:
            page_label = ev['page'] if ev['page'] is not None else 'n/a'
            print(f" - page={page_label}, sim={ev['similarity']}: {ev['chunk'][:180]}...")

# Example:
print_evidence_for_sp(phase1_results[0]['sp_file'])


Strategic plan: Bocconi_Piano_Strategico2021-2025&Vision2030.pdf

Rank 1: HORIZON-WIDERA-2023-ERA-01-06.pdf | final_score=0.6571
 - page=4, sim=0.7108: Horizon Europe - Work Programme 2023-2025 Widening participation and strengthening the European Research Area Part 11 - Page 103 of 193 faces in its capacity to better understand c...
 - page=3, sim=0.7098: Horizon Europe - Work Programme 2023-2025 Widening participation and strengthening the European Research Area Part 11 - Page 102 of 193 • Organise the material to be uploaded on th...
 - page=1, sim=0.7097: Horizon Europe - Work Programme 2023-2025 Widening participation and strengthening the European Research Area Part 11 - Page 100 of 193 support mutual learning and exchange of good...

Rank 2: HORIZON-WIDERA-2022-ERA-01-10.pdf | final_score=0.6067
 - page=2, sim=0.7172: Horizon Europe - Work Programme 2021-2022 Widening participation and strengthening the European Research Area Part 11 - Page 107 of 176  Identification of common 

## Phase 2 XAI Configuration

Define explainability settings and theme categories used to generate auditable rationale.


In [ ]:
XAI_TOP_THEMES = 4
XAI_EVIDENCE_PER_CALL = 3
XAI_MAX_GAPS = 4

GAP_MIN_CALL_SIGNAL = 0.2
GAP_MAX_SP_SIGNAL = 0.1


## Theme Lexicon

Map broad strategic themes to multilingual keywords for transparent, deterministic theme matching.


In [ ]:
THEME_LEXICON = {
    "sustainability_climate": [
        "sustainability", "sustainable", "climate", "green", "decarbonization",
        "circular economy", "energy transition", "biodiversity", "sdg",
        "sostenibilita", "sostenibile", "clima", "transizione energetica",
        "economia circolare", "decarbonizzazione", "biodiversita"
    ],
    "ai_data_digital": [
        "artificial intelligence", "ai", "machine learning", "data", "digital",
        "digitalization", "digitization", "big data", "cloud", "cybersecurity",
        "intelligenza artificiale", "apprendimento automatico", "dati", "digitale",
        "digitalizzazione", "transizione digitale"
    ],
    "internationalization_collaboration": [
        "international", "internationalization", "cross-border", "european", "consortium",
        "partnership", "collaboration", "mobility", "network", "joint",
        "internazionale", "internazionalizzazione", "europeo", "consorzio",
        "partenariato", "collaborazione", "mobilita", "rete", "congiunto"
    ],
    "innovation_transfer_industry": [
        "innovation", "technology transfer", "valorization", "startup", "spin-off",
        "industry", "industrial", "commercialization", "patent",
        "innovazione", "trasferimento tecnologico", "valorizzazione",
        "industria", "industriale", "brevett"
    ],
    "skills_education_capacity": [
        "skills", "competence", "training", "education", "lifelong learning",
        "upskilling", "reskilling", "capacity building", "doctoral", "curriculum",
        "competenze", "formazione", "istruzione", "apprendimento permanente", "dottorato"
    ],
    "inclusion_gender_social": [
        "inclusion", "equity", "equality", "gender", "diversity", "accessibility",
        "social impact", "vulnerable", "cohesion",
        "inclusione", "equita", "uguaglianza", "genere", "diversita",
        "accessibilita", "impatto sociale", "vulnerabili", "coesione"
    ],
    "research_infrastructure_excellence": [
        "research infrastructure", "infrastructure", "laboratory", "equipment",
        "excellence", "scientific excellence", "facility", "platform",
        "infrastruttura di ricerca", "infrastruttura", "laboratorio", "attrezzature",
        "eccellenza", "eccellenza scientifica", "piattaforma"
    ],
    "governance_policy_reform": [
        "governance", "reform", "policy", "regulation", "institutional",
        "coordination", "roadmap", "implementation", "monitoring", "evaluation",
        "riforma", "politica", "regolazione", "istituzionale",
        "coordinamento", "attuazione", "monitoraggio", "valutazione"
    ],
}


## XAI Builders

Convert evidence chunks into matched themes and actionable gaps for each recommended funding call.


In [ ]:
def build_call_xai(sp_text: str, call_row: Dict) -> Dict:
    return core_build_call_xai(
        sp_text=sp_text,
        call_row=call_row,
        theme_lexicon=THEME_LEXICON,
        xai_evidence_per_call=XAI_EVIDENCE_PER_CALL,
        xai_top_themes=XAI_TOP_THEMES,
        gap_min_call_signal=GAP_MIN_CALL_SIGNAL,
        gap_max_sp_signal=GAP_MAX_SP_SIGNAL,
        xai_max_gaps=XAI_MAX_GAPS,
    )



## Phase 2 Build

Attach explainability objects to each top-3 recommendation for every strategic plan.


In [ ]:
phase2_results = []

for result in phase1_results:
    sp = next((item for item in strategic_plans if item["source_file"] == result["sp_file"]), None)
    if sp is None:
        continue

    calls_with_xai = []
    for call in result["top_calls"]:
        call_copy = dict(call)
        call_copy["xai"] = build_call_xai(sp["text"], call_copy)
        calls_with_xai.append(call_copy)

    phase2_results.append({
        "sp_id": result["sp_id"],
        "sp_file": result["sp_file"],
        "raw_hits": result["raw_hits"],
        "top_calls": calls_with_xai,
    })

print(f"Built Phase 2 XAI objects for {len(phase2_results)} strategic plans")


## Phase 2 Summary Table

Show top-3 calls with matched themes and improvement actions in one compact output table.


In [ ]:
xai_rows = []

for result in phase2_results:
    for rank, call in enumerate(result["top_calls"], start=1):
        xai = call["xai"]
        themes = ", ".join([item["theme"] for item in xai["matched_themes"]])
        actions = " | ".join([item["action"] for item in xai["key_gaps"]])

        xai_rows.append({
            "sp_file": result["sp_file"],
            "rank": rank,
            "call_name": xai["call_name"],
            "final_score": xai["final_score"],
            "matched_themes": themes,
            "improvement_actions": actions,
        })

xai_df = pd.DataFrame(xai_rows).sort_values(["sp_file", "rank"]).reset_index(drop=True)
xai_df


## Phase 2 Evidence Preview

Inspect full evidence, matched themes, and gaps for one strategic plan recommendation set.


In [ ]:
def print_xai_for_sp(sp_file: str) -> None:
    matches = [row for row in phase2_results if row["sp_file"] == sp_file]
    if not matches:
        print(f"No Phase 2 result found for {sp_file}")
        return

    result = matches[0]
    print(f"Strategic plan: {result["sp_file"]}")

    for idx, call in enumerate(result["top_calls"], start=1):
        xai = call["xai"]
        print(f"\nRank {idx}: {xai["call_name"]} | final_score={xai["final_score"]}")
        print(f" Score breakdown: semantic={xai["score_breakdown"]["semantic_score"]}, coverage={xai["score_breakdown"]["coverage_score"]}, consistency={xai["score_breakdown"]["consistency_score"]}")

        print(" Matched themes:")
        for theme in xai["matched_themes"]:
            print(f"  - {theme["theme"]} (match={theme["match_strength"]}, call={theme["call_score"]}, sp={theme["sp_score"]})")

        print(" Key gaps:")
        if not xai["key_gaps"]:
            print("  - No critical gaps detected from current evidence.")
        else:
            for gap in xai["key_gaps"]:
                print(f"  - {gap["action"]} (gap={gap["gap_strength"]})")

        print(" Supporting chunks:")
        for ev in xai["supporting_chunks"]:
            page = ev["page"] if ev["page"] is not None else "n/a"
            print(f"  - source={ev["source_file"]}, page={page}, sim={ev["similarity"]}")
            print(f"    {ev["excerpt"][:220]}...")


print_xai_for_sp(phase2_results[0]["sp_file"])


## Phase 3 LLM Setup

Initialize local Ollama client configuration via shared llm_explainer module.


In [70]:
import importlib
import llm_explainer

importlib.reload(llm_explainer)
from llm_explainer import generate_summary

LLM_MODEL_NAME = "qwen3:0.6b"
PHASE3_TEMPERATURE = 0.1

print(f"LLM configured: {LLM_MODEL_NAME} (llm_explainer reloaded)")


LLM configured: qwen3:0.6b (llm_explainer reloaded)


## Phase 3 Prompt and Policy

Configure deterministic, evidence-grounded LLM summaries for executive stakeholders.


In [71]:
import json
from datetime import datetime, timezone

PHASE3_LANGUAGE = "English"
PHASE3_MIN_CITATIONS_PER_CALL = 2
PHASE3_TARGET_ACTIONS = 5
PHASE3_MAX_RETRY = 0  # stop on failure

PHASE3_OUTPUT_DIR = PROJECT_ROOT / "outputs" / "phase3"
PHASE3_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PHASE3_CONFIDENCE_RULE = (
    "Assign confidence as High/Medium/Low. If evidence is weak or citations are missing, downgrade confidence."
)

print(f"Phase 3 output dir: {PHASE3_OUTPUT_DIR}")


Phase 3 output dir: /home/lburmazevic/Projects/EY-AI-Techniques-Solution/outputs/phase3


## Phase 3 Builders

Build grounded prompts from Phase 2 outputs and parse model responses.


In [72]:
def _build_phase3_payload(sp_row: Dict) -> Dict:
    calls_payload = []
    for rank, call in enumerate(sp_row.get("top_calls", []), start=1):
        xai = call.get("xai", {})
        supporting = []
        for ev in xai.get("supporting_chunks", []):
            supporting.append(
                {
                    "source_file": ev.get("source_file"),
                    "page": ev.get("page"),
                    "similarity": ev.get("similarity"),
                    "excerpt": ev.get("excerpt", ""),
                }
            )
        calls_payload.append(
            {
                "rank": rank,
                "call_name": xai.get("call_name", call.get("call_name")),
                "scores": {
                    "final_score": xai.get("final_score", call.get("final_score")),
                    "semantic_score": xai.get("score_breakdown", {}).get("semantic_score", call.get("semantic_score")),
                    "coverage_score": xai.get("score_breakdown", {}).get("coverage_score", call.get("coverage_score")),
                    "consistency_score": xai.get("score_breakdown", {}).get("consistency_score", call.get("consistency_score")),
                },
                "matched_themes": xai.get("matched_themes", []),
                "key_gaps": xai.get("key_gaps", []),
                "supporting_chunks": supporting,
            }
        )
    return {
        "sp_id": sp_row.get("sp_id"),
        "sp_file": sp_row.get("sp_file"),
        "raw_hits": sp_row.get("raw_hits"),
        "top_calls": calls_payload,
    }
def _build_phase3_prompt(payload: Dict) -> str:
    schema = {
        "sp_id": "string",
        "sp_file": "string",
        "confidence": "High | Medium | Low",
        "executive_summary": "2-4 sentences, stakeholder style",
        "top_calls": [
            {
                "rank": "integer",
                "call_name": "string",
                "why_match": "2-3 sentences",
                "citations": [
                    {"source_file": "string", "page": "int|null"},
                    {"source_file": "string", "page": "int|null"}
                ],
                "confidence": "High | Medium | Low"
            }
        ],
        "improvement_actions": [
            "up to 5 action bullets based only on provided key_gaps/evidence"
        ]
    }
    instructions = f"""
You are an executive advisor. Write in {PHASE3_LANGUAGE}.
Use only the provided evidence. Do not invent facts.
Each top call must include at least {PHASE3_MIN_CITATIONS_PER_CALL} citations from provided chunks.
{PHASE3_CONFIDENCE_RULE}
Return two sections exactly:
<JSON>
{{valid JSON matching schema}}
</JSON>
<TEXT>
Readable stakeholder briefing with headings:
- Top 3 Calls
- Why They Match
- Recommended Focus Areas
- Confidence Note
</TEXT>
""".strip()
    return (
        instructions
        + "\n\nJSON schema:\n"
        + json.dumps(schema, ensure_ascii=True, indent=2)
        + "\n\nInput payload:\n"
        + json.dumps(payload, ensure_ascii=True, indent=2)
    )
def _parse_llm_response(text: str) -> Dict:
    json_start = text.find("<JSON>")
    json_end = text.find("</JSON>")
    text_start = text.find("<TEXT>")
    text_end = text.find("</TEXT>")
    if json_start == -1 or json_end == -1:
        raise ValueError("Model response missing <JSON> section")
    if text_start == -1 or text_end == -1:
        raise ValueError("Model response missing <TEXT> section")
    json_block = text[json_start + len("<JSON>"):json_end].strip()
    text_block = text[text_start + len("<TEXT>"):text_end].strip()
    parsed = json.loads(json_block)
    return {
        "json": parsed,
        "text": text_block,
        "json_block_raw": json_block,
    }


## Phase 3 LLM Generation

Generate one grounded summary per SP, stop immediately on any failure.


In [74]:
def generate_phase3_for_sp(sp_row: Dict) -> Dict:
    llm = generate_summary(
        strategic_plan=sp_row,
        model=LLM_MODEL_NAME,
        temperature=PHASE3_TEMPERATURE,
        min_citations_per_call=PHASE3_MIN_CITATIONS_PER_CALL,
        language=PHASE3_LANGUAGE,
    )

    now_iso = datetime.now(timezone.utc).isoformat()

    result = {
        "sp_id": sp_row.get("sp_id"),
        "sp_file": sp_row.get("sp_file"),
        "model_name": LLM_MODEL_NAME,
        "timestamp": now_iso,
        "temperature": PHASE3_TEMPERATURE,
        "prompt": llm["prompt"],
        "raw_response_text": llm["raw_response_text"],
        "structured": llm["structured"],
        "stakeholder_text": llm["stakeholder_text"],
    }

    return result


phase3_results = []
for sp_row in phase2_results:
    phase3_results.append(generate_phase3_for_sp(sp_row))

print(f"Generated Phase 3 summaries for {len(phase3_results)} strategic plans")
phase3_results[0]["structured"] if phase3_results else {}



Generated Phase 3 summaries for 6 strategic plans


{'sp_id': 'Bocconi_Piano_Strategico2021-2025&Vision2030',
 'sp_file': 'Bocconi_Piano_Strategico2021-2025&Vision2030.pdf',
 'confidence': 'Medium',
 'executive_summary': 'The Bocconi Strategy focuses on three pillars: internationalization, innovation, and sustainability. Top 3 Calls align with these themes, with PRIN 2022 PNRR and Ecosistemi PNRR highlighting strategic partnerships and technological transfer, while Partenariati PNRR emphasizes AI and social inclusion.',
 'top_calls': [{'rank': 1,
   'call_name': 'PRIN 2022 PNRR.pdf',
   'why_match': 'PRIN 2022 PNRR emphasizes international collaboration and innovation transfer, as evidenced by its alignment with sustainability and AI initiatives.',
   'citations': [{'source_file': 'PRIN 2022 PNRR.pdf',
     'page': 32,
     'similarity': 0.6981,
     'excerpt': "Ministero dell'Università e della Ricerca Direzione Generale della Ricerca Ufficio III 32 procedura di finanziamento, di avere beneficiato dei fondi dell'Unione Europea – Next G

## Phase 3 Validation and Exports

Validate citation policy and export Phase 3 outputs for frontend/report usage.


In [75]:
validation_rows = []

for item in phase3_results:
    structured = item.get("structured", {})
    top_calls = structured.get("top_calls", [])

    for call in top_calls:
        citations = call.get("citations", [])
        validation_rows.append(
            {
                "sp_file": item.get("sp_file"),
                "call_name": call.get("call_name"),
                "rank": call.get("rank"),
                "citation_count": len(citations),
                "citation_ok": len(citations) >= PHASE3_MIN_CITATIONS_PER_CALL,
                "confidence": call.get("confidence"),
            }
        )

phase3_validation_df = pd.DataFrame(validation_rows)
if not phase3_validation_df["citation_ok"].all():
    raise ValueError("Citation policy failed: at least one top-call has fewer than 2 citations")

phase3_structured_path = PHASE3_OUTPUT_DIR / "phase3_structured_results.json"
phase3_text_path = PHASE3_OUTPUT_DIR / "phase3_stakeholder_texts.json"
phase3_prompts_path = PHASE3_OUTPUT_DIR / "phase3_prompts_and_raw_outputs.json"
phase3_validation_path = PHASE3_OUTPUT_DIR / "phase3_validation.csv"

phase3_structured_path.write_text(
    json.dumps([x["structured"] for x in phase3_results], ensure_ascii=True, indent=2)
)
phase3_text_path.write_text(
    json.dumps([
        {"sp_id": x["sp_id"], "sp_file": x["sp_file"], "stakeholder_text": x["stakeholder_text"]}
        for x in phase3_results
    ], ensure_ascii=True, indent=2)
)
phase3_prompts_path.write_text(
    json.dumps([
        {
            "sp_id": x["sp_id"],
            "sp_file": x["sp_file"],
            "model_name": x["model_name"],
            "timestamp": x["timestamp"],
            "prompt": x["prompt"],
            "raw_response_text": x["raw_response_text"],
        }
        for x in phase3_results
    ], ensure_ascii=True, indent=2)
)
phase3_validation_df.to_csv(phase3_validation_path, index=False)

print(f"Saved: {phase3_structured_path}")
print(f"Saved: {phase3_text_path}")
print(f"Saved: {phase3_prompts_path}")
print(f"Saved: {phase3_validation_path}")

phase3_validation_df



Saved: /home/lburmazevic/Projects/EY-AI-Techniques-Solution/outputs/phase3/phase3_structured_results.json
Saved: /home/lburmazevic/Projects/EY-AI-Techniques-Solution/outputs/phase3/phase3_stakeholder_texts.json
Saved: /home/lburmazevic/Projects/EY-AI-Techniques-Solution/outputs/phase3/phase3_prompts_and_raw_outputs.json
Saved: /home/lburmazevic/Projects/EY-AI-Techniques-Solution/outputs/phase3/phase3_validation.csv


,sp_file,call_name,rank,citation_count,citation_ok,confidence
0,Bocconi_Piano_Strategico2021-2025&Vision2030.pdf,PRIN 2022 PNRR.pdf,1,2,True,Medium
1,Bocconi_Piano_Strategico2021-2025&Vision2030.pdf,Ecosistemi dell'Innovazione PNRR.pdf,2,2,True,Medium
2,Bocconi_Piano_Strategico2021-2025&Vision2030.pdf,Partenariati Estesi PNRR.pdf,3,2,True,Medium
3,LUM_PIANO-STRATEGICO-DATENEO-2021-2025.pdf,Ecosistemi dell'Innovazione PNRR.pdf,1,2,True,Medium
4,LUM_PIANO-STRATEGICO-DATENEO-2021-2025.pdf,Partenariati Estesi PNRR.pdf,2,2,True,Medium
5,LUM_PIANO-STRATEGICO-DATENEO-2021-2025.pdf,PRIN PNRR 2022 NextGen.pdf,3,2,True,Medium
6,Naples_Federico_Piano_strategico_2021_2026.pdf,PRIN 2022 PNRR.pdf,1,2,True,Medium
7,Naples_Federico_Piano_strategico_2021_2026.pdf,PRIN PNRR 2022 NextGen.pdf,2,2,True,Medium
8,Naples_Federico_Piano_strategico_2021_2026.pdf,Partenariati Estesi PNRR.pdf,3,2,True,Medium
9,Piano Strategico Luiss 2021-2024 (per sito).pdf,PRIN 2022 PNRR.pdf,1,2,True,High
